# P0 — pykrx 데이터 깊이 프로브 (API 키 불필요)

**목적:** 한국 가격·시총·PBR이 각각 **몇 년까지** 닿는지 확인한다.
이게 표본 기간을 좌우한다 — 특히 PBR이 2015년 이전으로 닿으면, OpenDART(재무제표 2015+) 없이도
B/M(가치 팩터)을 더 길게 뽑는 우회로가 생긴다.

> 최근 pykrx는 일부 데이터에 KRX 계정 로그인이 필요할 수 있다. 함수가 빈 결과/오류를 내면
> 환경변수 `KRX_ID`, `KRX_PW`를 설정한 뒤 다시 돌려보자.
>
> 결과를 본 다음, 각 칸의 **가장 이른 날짜와 종목 수**를 알려줘. 그걸로 표본 기간을 확정한다.

## 준비

`pip install pykrx pandas`

In [1]:
from pykrx import stock
import pandas as pd

TODAY = pd.Timestamp.today().strftime('%Y%m%d')
SAMSUNG = '005930'   # 삼성전자 (1975년 상장 — 깊은 역사 기준 종목)
print('오늘:', TODAY)

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
오늘: 20260629


## 1) 가격(OHLCV) 깊이

삼성전자 일별 종가를 1995년부터 요청 → 실제로 데이터가 시작되는 날짜를 본다.

In [2]:
px = stock.get_market_ohlcv_by_date('19950101', TODAY, SAMSUNG)
print('가격 데이터:', px.index.min(), '~', px.index.max(), '/', len(px), '행')
px.head(2)

가격 데이터: 2014-04-03 00:00:00 ~ 2026-06-26 00:00:00 / 3000 행


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2014-04-03,27020,27900,27020,27800,381976,NaN
2014-04-04,27540,27940,27540,27600,368007,-0.719424


## 2) 시가총액 깊이 (사이즈 팩터용)

In [3]:
cap = stock.get_market_cap_by_date('19950101', TODAY, SAMSUNG)
print('시총 데이터:', cap.index.min(), '~', cap.index.max(), '/', len(cap), '행')
cap.head(2)

Error occurred in get_market_cap_by_date: Expecting value: line 1 column 1 (char 0)
시총 데이터: nan ~ nan / 0 행


""


## 3) 펀더멘털(PBR·BPS) 깊이 ← B/M 우회로의 핵심

KRX가 직접 공시하는 PBR이 몇 년부터 유효한지. 여기가 2015년 이전으로 닿으면 표본이 길어진다.

In [4]:
fun = stock.get_market_fundamental_by_date('19950101', TODAY, SAMSUNG)
print('펀더멘털 컬럼:', list(fun.columns))
print('전체 범위:', fun.index.min(), '~', fun.index.max())
valid = fun[fun['PBR'] > 0]
print('PBR 유효 시작:', valid.index.min() if len(valid) else '없음')
fun.head(2)

Error occurred in get_market_fundamental_by_date: Expecting value: line 1 column 1 (char 0)
펀더멘털 컬럼: []
전체 범위: nan ~ nan


KeyError: 'PBR'

## 4) 과거 유니버스(종목 수) — KOSPI / KOSDAQ

여러 과거 날짜에서 종목 리스트가 제대로 나오는지 (유니버스 구성이 그 시점에 가능한지).

In [ ]:
for d in ['20001229', '20051230', '20101230', '20151230', '20201230']:
    try:
        k = len(stock.get_market_ticker_list(d, 'KOSPI'))
        q = len(stock.get_market_ticker_list(d, 'KOSDAQ'))
        print(f'{d}:  KOSPI {k}개,  KOSDAQ {q}개')
    except Exception as e:
        print(f'{d}:  실패 — {type(e).__name__}: {e}')

## 5) 특정 날짜 전체 시장 PBR (B/M 정렬이 가능한가)

한 날짜에 시장 전체의 PBR을 한 번에 받을 수 있어야 25개 바구니로 정렬할 수 있다.

In [ ]:
for d in ['20151230', '20101230', '20051230']:
    try:
        f = stock.get_market_fundamental_by_ticker(d, 'KOSPI')
        nvalid = int((f['PBR'] > 0).sum())
        print(f'{d} KOSPI:  {len(f)}종목 중 PBR 유효 {nvalid}개')
    except Exception as e:
        print(f'{d}:  실패 — {type(e).__name__}: {e}')

## 결과 해석 가이드

- **1·2번(가격·시총)**: 시작 연도 = 시장팩터(Mkt−RF)·사이즈(SMB)의 최대 가능 표본.
- **3·5번(PBR)**: 시작 연도 = B/M·HML의 최대 가능 표본. **2015년보다 이르면** OpenDART 없이도 가치 팩터를 더 길게 뽑을 수 있다는 뜻(단 KRX의 PBR은 단순 장부가 기준이라 FF 충실판보다 정의가 거칠다 — v0.1 한계로 명기).
- **4번(종목 수)**: 그 시점 유니버스가 충분히 크면 정렬이 의미 있다.

→ 각 칸의 **가장 이른 날짜 + 종목 수**를 가져와줘. 그걸로 표본 기간을 확정하고 PRD v0.3로 올린다.